# Tutorial: GovernAI Support Workflow (Real LLM + Approval Gate)

This notebook runs a deterministic, governed support workflow:
1. Validate user input
2. Fetch customer profile
3. Draft reply with a real OpenAI call
4. Pause for approval before send
5. Resume and complete with full audit events


## Outline

- Setup and API key loading from adjacent `letstravel/.env`
- Define typed tools + policy + deterministic workflow
- Execute run -> approval interruption -> resume
- Inspect artifacts and audit trail


In [ ]:
import asyncio
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

REPO_ROOT = Path.cwd().resolve().parent.parent if "notebooks" in str(Path.cwd()) else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / '.env')
print('Loaded:', REPO_ROOT / '.env')
print('OPENAI_API_KEY present:', bool(os.getenv('OPENAI_API_KEY')))
print('OPENAI model:', os.getenv('OPENAI_MODEL', 'gpt-4o-mini'))


## Step 1 - Define Governed Workflow Components


In [ ]:
from pydantic import BaseModel

from governai import (
    ApprovalDecision,
    ApprovalDecisionType,
    PolicyDecision,
    Workflow,
    policy,
    step,
    tool,
)


def make_llm() -> ChatOpenAI:
    api_key = os.getenv('OPENAI_API_KEY')
    if not api_key:
        raise RuntimeError('Missing OPENAI_API_KEY in local .env')
    return ChatOpenAI(
        model=os.getenv('OPENAI_MODEL', 'gpt-4o-mini'),
        api_key=api_key,
        temperature=0.2,
    )


class SupportIn(BaseModel):
    customer_id: str
    customer_email: str
    issue: str


class ValidateOut(BaseModel):
    customer_id: str
    customer_email: str
    issue: str


class CustomerOut(BaseModel):
    customer_id: str
    customer_email: str
    customer_name: str
    tier: str
    issue: str


class DraftOut(BaseModel):
    subject: str
    body: str
    to_email: str


class SendOut(BaseModel):
    sent: bool
    provider_message_id: str


@tool(name='support.validate', input_model=SupportIn, output_model=ValidateOut)
async def validate_input(ctx, data: SupportIn) -> ValidateOut:
    if len(data.issue.strip()) < 6:
        raise ValueError('Issue must be at least 6 characters')
    return ValidateOut(**data.model_dump())


@tool(name='customer.fetch', input_model=ValidateOut, output_model=CustomerOut)
async def fetch_customer(ctx, data: ValidateOut) -> CustomerOut:
    return CustomerOut(
        customer_id=data.customer_id,
        customer_email=data.customer_email,
        customer_name='Alex Doe',
        tier='gold',
        issue=data.issue,
    )


@tool(name='reply.draft', input_model=CustomerOut, output_model=DraftOut, capabilities=['llm.invoke'])
async def draft_reply(ctx, data: CustomerOut) -> DraftOut:
    llm = make_llm()
    result = llm.invoke([
        SystemMessage(content='You are a precise customer support assistant.'),
        HumanMessage(content=(
            f"Customer: {data.customer_name}\n"
            f"Tier: {data.tier}\n"
            f"Issue: {data.issue}\n"
            'Write a concise, professional support reply with next steps.'
        )),
    ])
    return DraftOut(
        subject=f'Re: Support for {data.customer_id}',
        body=result.content,
        to_email=data.customer_email,
    )


@tool(
    name='reply.send',
    input_model=DraftOut,
    output_model=SendOut,
    side_effect=True,
    requires_approval=True,
    capabilities=['email.send'],
)
async def send_reply(ctx, data: DraftOut) -> SendOut:
    return SendOut(sent=True, provider_message_id=f'mock-{data.to_email}')


@policy('deny_send_without_explicit_approval')
def deny_send_without_explicit_approval(ctx) -> PolicyDecision:
    if ctx.tool_name != 'reply.send':
        return PolicyDecision(allow=True)
    approved_steps = set(ctx.metadata.get('approved_steps', []))
    if ctx.step_name not in approved_steps:
        return PolicyDecision(allow=False, reason='External send requires approval')
    return PolicyDecision(allow=True)


class SupportGovernedFlow(Workflow[SupportIn, SendOut]):
    validate = step('validate', tool=validate_input).then('fetch_customer')
    fetch_customer = step('fetch_customer', tool=fetch_customer).then('draft_reply')
    draft_reply = step('draft_reply', tool=draft_reply).then('send_reply')
    send_reply = step('send_reply', tool=send_reply).then_end()


## Step 2 - Run, Interrupt for Approval, Resume


In [ ]:
async def run_demo(issue: str):
    flow = SupportGovernedFlow()
    flow.runtime.policy_engine.register(deny_send_without_explicit_approval)

    first = await flow.run(
        SupportIn(
            customer_id='cust-42',
            customer_email='customer@example.com',
            issue=issue,
        )
    )
    print('Status after first run:', first.status)
    print('Pending approval:', bool(first.pending_approval))

    resumed = await flow.resume(
        first.run_id,
        ApprovalDecision(decision=ApprovalDecisionType.APPROVE, decided_by='notebook-user'),
    )
    print('Final status:', resumed.status)
    print('Final send artifact:', resumed.artifacts.get('send_reply'))

    print('\nAudit events:')
    for event in flow.runtime.audit_emitter.events:
        print(f"- {event.event_type.value:28s} step={event.step_name} payload={event.payload}")

    return resumed


state = asyncio.run(run_demo('I was charged twice for last month and need help.'))
print('\nDraft preview:\n', state.artifacts['draft_reply']['body'][:600])
